In [0]:
%pip install requests

In [0]:
%restart_python

# Step 1: Fetch the data

## Download to `/tmp` Using requests library ❌
Fails because moving files from `/tmp` to a volume is apparently forbidden


## Download directly to dbfs using `dbutils.fs.cp` ❌
Fails with `ExecutionError: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /FileStore/nppes/raw/2026-03-27/nppes.zip SQLSTATE: 56038` --> limitation in the free edition >> why we create volumes

In [0]:
# "Blocked by Unity Catalog security"
dbutils.fs.cp(
    "https://download.cms.gov/nppes/NPPES_Data_Dissemination_March_2026_V2.zip",
    "dbfs:/FileStore/nppes/raw/2026-03-27/nppes.zip"
)

## Download zip file directly to volume ✅
Zip file is successfully downloaded to `dbfs:/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip`

In [0]:
from datetime import datetime

#run_date = datetime.today().strftime("%Y-%m-%d")

base_path = f"/Volumes/bronze_dev/nppes_npi_registry/raw/"
zip_path = base_path + "nppes.zip"
extract_path = base_path + "unzipped/"

In [0]:
dbutils.fs.mkdirs(base_path)

dbutils.fs.cp(
    "https://download.cms.gov/nppes/NPPES_Data_Dissemination_March_2026_V2.zip",
    zip_path
)

Databricks view

In [0]:
dbutils.fs.ls(base_path)

In [0]:
display(dbutils.fs.ls(base_path))

# Unzip the data

In [0]:
print(f"{zip_path=}")
print(f"{extract_path=}")

### 1) In Python with `zipfile` library -- WORKING!

Fails with `OSError: [Errno 5] Input/output error: '/dbfs/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip'`

ChatGPT says this fails because "Python (via /dbfs/Volumes/...) is trying to read a large file through the FUSE layer, and it’s failing."

In [0]:
import zipfile

with zipfile.ZipFile(zip_path, 'r'/Volumes/bronze_dev/nppes_npi_registry/raw/unzipped/) as zip_ref:
    zip_ref.extractall(extract_path)

### 2) in bash ❌

Fails because OS-level access issues

Per ChatGPT: In many Unity Catalog environments: /dbfs/Volumes/... is NOT supported for OS-level access (%sh, Python file IO, etc.)

In [0]:
%sh
ls -lh /dbfs/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/

In [0]:
%sh
unzip /dbfs/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip \
     -d /dbfs/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/unzipped/

### 3) Using spark, `zipfile` and `io` libraries

In [0]:
import zipfile
import io

# Read ZIP as binary via Spark
binary_df = spark.read.format("binaryFile").load(zip_path)

# Collect binary content (safe: single file)
zip_bytes = binary_df.select("content").first()[0]

# Open ZIP in memory
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        file_content = z.read(name)
        
        dbutils.fs.put(
            extract_path + name,
            file_content,
            overwrite=True
        )

## Unzip using Spark

In [0]:

print(f"{zip_path=}")

print(f"{extract_path=}")

### 1) `zipfile`, `io`, and `dbutils.fs.put` ❌
Failed with `[FAILED_READ_FILE.NO_HINT] Error while reading file dbfs:/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip.  SQLSTATE: KD001`

In [0]:
import zipfile
import io

# Read ZIP as binary via Spark
binary_df = spark.read.format("binaryFile").load(zip_path)

# Collect binary content (safe: single file)
zip_bytes = binary_df.select("content").first()[0]

# Open ZIP in memory
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        file_content = z.read(name)
        
        dbutils.fs.put(
            extract_path + name,
            file_content,
            overwrite=True
        )

### with spark

Failed with `[FAILED_READ_FILE.NO_HINT] Error while reading file dbfs:/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip.  SQLSTATE: KD001`

In [0]:
import zipfile
import io
from pyspark.sql import SparkSession

zip_path = "/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip"
extract_base = "/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/unzipped/"

# Read the ZIP as binary via Spark
binary_df = spark.read.format("binaryFile").load(zip_path)
zip_bytes = binary_df.select("content").first()[0]

# Extract each file into Unity Catalog volume
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        file_content = z.read(name)
        dbutils.fs.put(extract_base + name, file_content, overwrite=True)

## Using `zipfile` library

`OSError: [Errno 5] Input/output error: '/dbfs/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip'`

In [0]:
import requests

url = "https://download.cms.gov/nppes/NPPES_Data_Dissemination_March_2026_V2.zip"
local_path = "/tmp/nppes.zip"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
            if chunk:
                f.write(chunk)
import os

print(os.path.exists("/tmp/nppes.zip"))
size_bytes = os.path.getsize("/tmp/nppes.zip")
size_mb = size_bytes / (1024 * 1024)

print(f"zip file size: {size_mb:.2f} MB")

import zipfile

with zipfile.ZipFile("/tmp/nppes.zip", "r") as z:
    print(len(z.namelist()))
    print(z.namelist()[:10])  # preview first 10 files

# Copy the downloaded file to Unity Catalog Volume
dbutils.fs.cp(
    "file:/tmp/nppes.zip",
    "/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip"
)